
# Silver — PESQUISA DE URA

Atualiza a Silver de PESQUISA consumindo o **Change Data Feed** da Bronze como
**fonte de streaming** (`Trigger.AvailableNow`) e aplicando **MERGE** idempotente via
`foreachBatch`. O **checkpoint** do stream controla o progresso, com garantia de
*exactly-once*.

**Fluxo**
1. Lê o CDF da Bronze como stream, a partir do checkpoint.
2. Filtra `insert`, faz parse do `body` (JSON) e deriva o período.
3. Aplica **MERGE** por `ID_CHAM`.



## Parâmetros


In [ ]:
# ===================== PARÂMETROS (Widgets) =====================
import sys

sys.path.append("/Workspace/Repos/data_master/Databricks/lib")

dbutils.widgets.text("catalog", "prd")
dbutils.widgets.text("bronze_schema", "b_dm_callcenter")
dbutils.widgets.text("silver_schema", "s_dm_callcenter")
dbutils.widgets.text("bronze_table", "surveys_once")
dbutils.widgets.text("silver_table", "tabe_pesq_ura")
dbutils.widgets.text("checkpoint_base", "/Volumes/prd/s_dm_callcenter/checkpoints/silver")

CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")
BRONZE_TABLE  = dbutils.widgets.get("bronze_table")
SILVER_TABLE  = dbutils.widgets.get("silver_table")

BRONZE_FQN = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}"
SILVER_FQN = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"
CHECKPOINT = f"{dbutils.widgets.get('checkpoint_base').rstrip('/')}/{SILVER_TABLE}"

from pyspark.sql import functions as F, types as T
from transforms import SilverStream

print("Bronze:", BRONZE_FQN)
print("Silver:", SILVER_FQN)
print("Checkpoint:", CHECKPOINT)


## Transformação


In [0]:
def transform(df_raw):
    # Schema do JSON
    schema = T.StructType([
        T.StructField("id_chamada", T.StringType(),  True),
        T.StructField("id_pesquisa", T.StringType(),  True),
        T.StructField("data_envio", T.DateType(), True),
        T.StructField("nota", T.IntegerType(), True)
    ])

    rename = {
        'id_chamada':     'ID_CHAM',
        'id_pesquisa':    'ID_PESQ',
        'data_envio':     'DT_ENVI',
        'nota':           'VL_NOTA'
    }

    df = (df_raw
          .withColumn("body", F.from_json(F.col("body"), schema))
          .filter(F.col("body").isNotNull())
          .withColumn("_cv", F.col("_commit_version").cast("long"))
          .withColumn("_ct", F.col("_commit_timestamp").cast("timestamp"))
          .select("body.*", "_cv", "_ct")
    )

    # Renomeia as colunas
    df = df.select([F.col(c).alias(rename.get(c, c)) for c in df.columns])

    # Colunas de DATA do evento e carga
    df = (df
          .withColumn("CD_PERI", F.date_format(F.col("DT_ENVI"), "yyyyMM").cast("int"))
          .withColumn("DH_REFE_CRGA", F.current_timestamp())
    )
    return df



## ▶️ Execução
Stream do CDF (`AvailableNow`) → `transform` → `MERGE` idempotente por `ID_CHAM` via
`foreachBatch`. O **checkpoint** controla o progresso.


In [ ]:
# Upsert incremental via streaming (AvailableNow) + foreachBatch(MERGE).
stream = SilverStream(spark)
stream.run(
    source_fqn=BRONZE_FQN,
    target_fqn=SILVER_FQN,
    transform=transform,
    keys=["ID_CHAM"],
    checkpoint_location=CHECKPOINT,
    cluster_by=["ID_PESQ"],
)
print(f"[OK] Silver atualizada → {SILVER_FQN}")